# ShadowWork-SHIFT | Complete historical evidence publication

**Run on Kaggle CPU with Internet enabled.** Attach the ZIP `ShadowWork_SHIFT_Publication_Evidence.zip` as a Kaggle input dataset; set write-enabled `HF_TOKEN` in Kaggle Secrets. This is an **addendum** after the main successful SW12/SW17 upload, not a replacement. It uploads missing SW13–SW16 archives/summaries to the existing public dataset repository without training anything.


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2"


In [ ]:
# ShadowWork-SHIFT | Add verified SW13–SW16 historical archives to the public dataset.
# CPU-only publication. No training, model loading, evaluation, or experiment changes.

from pathlib import Path
from collections import Counter
import json
import hashlib
import zipfile
import shutil
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

DATASET_ID = "i-am-shaurya05/shadowwork-shift-datasets"
ARCHIVE_NAME = "ShadowWork_SHIFT_Publication_Evidence.zip"
SOURCE_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/shift_historical_publication")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

matches = sorted(SOURCE_ROOT.rglob(ARCHIVE_NAME))
assert len(matches) == 1, (
    "Attach the verified historical-evidence ZIP to this Kaggle notebook as an input dataset.", matches
)
archive_path = matches[0]
assert archive_path.is_file()
assert archive_path.stat().st_size > 4_000_000
print("Input archive:", archive_path)

export_dir = OUTPUT_ROOT / "archived_evidence"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

# Preserve the original five SW13–SW16 result ZIPs. Include plain-text summaries
# so reviewers can inspect metrics without downloading an archive.
expected_source_archives = {
    "ShadowWork_13_Live_Results.zip",
    "ShadowWork_14_Controller_Repair_Results1.zip",
    "ShadowWork_15_Cross_Protocol_Recovery_Results.zip",
    "ShadowWork_16_Unified_Recovery_Results.zip",
    "ShadowWork_16_Budget_Regression_Results.zip",
}
selected_prefixes = (
    "source_archives/",
    "sw13_live/",
    "sw14_controller/",
    "sw15_cross_protocol/",
    "sw16_unified/",
    "sw16_budget/",
)
with zipfile.ZipFile(archive_path) as source:
    assert source.testzip() is None, "Corrupted historical evidence archive"
    member_names = set(source.namelist())
    manifest = json.loads(source.read("SHA256_MANIFEST.json"))["sha256"]
    assert set(manifest) == member_names - {"SHA256_MANIFEST.json"}
    for name, expected_hash in manifest.items():
        assert hashlib.sha256(source.read(name)).hexdigest() == expected_hash, name
    present = {
        Path(name).name for name in member_names
        if name.startswith("source_archives/") and name.endswith(".zip")
    }
    assert expected_source_archives <= present
    for name in sorted(member_names):
        if not name.startswith(selected_prefixes) or name.endswith("/"):
            continue
        path = Path(name)
        assert not path.is_absolute() and ".." not in path.parts
        destination = export_dir / path
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(source.read(name))

# The original files are reproduced byte-for-byte. The Kaggle source ZIP
# can be kept in Project storage; do not upload redundant SW12 files.
rows = {
    p.relative_to(export_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted(export_dir.rglob("*")) if p.is_file()
}
(export_dir / "HISTORICAL_SHA256_MANIFEST.json").write_text(
    json.dumps(rows, indent=2, sort_keys=True) + "\n"
)
(export_dir / "README.md").write_text("""# ShadowWork-SHIFT | Archived SW13–SW16 evidence

These are the original source result ZIP archives and inspectable summary files
from the SW13 live-controller evaluation, SW14 controller repair, SW15
cross-protocol recovery, SW16 unified controller and SW16 budget-regression runs.

They are **historical synthetic experiments**, not independent external
benchmarks. Their task counts and denominators are not to be combined with
SW12's sealed 64 tasks or SW17's sealed 80 tasks.

Source notebooks, limitations and result interpretation:
https://github.com/githubshaurya/ShadowWork-SHIFT

Exact SHA-256 fingerprints of every file in this directory are recorded in
HISTORICAL_SHA256_MANIFEST.json. No model training was performed here.
""")
print("Prepared archived evidence files:", len(rows))
print("Original source ZIP files:", len(expected_source_archives))

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Set Kaggle Secret HF_TOKEN with write permission"
hub = HfApi(token=HF_TOKEN)
assert hub.whoami()["name"] == "i-am-shaurya05"
hub.upload_folder(
    repo_id=DATASET_ID,
    repo_type="dataset",
    folder_path=str(export_dir),
    path_in_repo="archived_evidence",
    commit_message="Publish verified SW13-SW16 historical evidence ZIPs and summaries",
)
# Anonymously confirm every source ZIP and the SHA manifest are public.
public = HfApi(token=False).dataset_info(DATASET_ID, token=False)
remote_files = {item.rfilename for item in public.siblings}
needed = {
    f"archived_evidence/source_archives/{name}"
    for name in expected_source_archives
}
needed |= {
    "archived_evidence/README.md",
    "archived_evidence/HISTORICAL_SHA256_MANIFEST.json",
}
assert not public.private and needed <= remote_files, sorted(needed - remote_files)
print("PASS: Historical SW13–SW16 evidence publicly uploaded without retraining.")
print("https://huggingface.co/datasets/" + DATASET_ID)
